In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [3]:
if torch.cuda.is_available():
    # Print the name of the GPU
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    # Print the number of available CUDA devices
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    print("No GPU available. Training will run on CPU.")


No GPU available. Training will run on CPU.


In [4]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4

In [5]:
w_q = nn.Linear(d_model, d_model)
w_k = nn.Linear(d_model, d_model)
w_v = nn.Linear(d_model, d_model)
w_o = nn.Linear(d_model, d_model)

In [6]:
dummy_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.5991, 0.1468, 0.3029, 0.7924, 0.8305, 0.4818, 0.5789, 0.2537],
         [0.3113, 0.1703, 0.8655, 0.2709, 0.0793, 0.6347, 0.6968, 0.0773],
         [0.0745, 0.1771, 0.7588, 0.7103, 0.8310, 0.1615, 0.3085, 0.7847]],

        [[0.4042, 0.4617, 0.5945, 0.5623, 0.4233, 0.1586, 0.3845, 0.0034],
         [0.2339, 0.6837, 0.3431, 0.8735, 0.5617, 0.1242, 0.5305, 0.2417],
         [0.9068, 0.0084, 0.5761, 0.4857, 0.8672, 0.6024, 0.6004, 0.7621]]])

In [7]:
q = w_q(dummy_input)
k = w_k(dummy_input)
v = w_v(dummy_input)

In [8]:
print(q.shape)
print("(batch_size, num_tokens, key_dimension)")
q

torch.Size([2, 3, 8])
(batch_size, num_tokens, key_dimension)


tensor([[[ 0.0868, -0.3236,  0.1107,  0.1021,  0.3104, -0.5453,  0.3435,
           0.1907],
         [ 0.3221, -0.5190,  0.1238,  0.2708,  0.1253, -0.1416, -0.1077,
           0.0083],
         [-0.0153, -0.5436,  0.5184,  0.4468,  0.1432, -0.4153,  0.1642,
           0.2217]],

        [[ 0.3290, -0.5475,  0.2260,  0.2035, -0.0223, -0.2448,  0.0392,
           0.2375],
         [ 0.0569, -0.4688,  0.3325,  0.3095,  0.0971, -0.3771,  0.1712,
           0.1964],
         [ 0.0776, -0.4909,  0.0411,  0.2400,  0.3649, -0.5768,  0.4504,
           0.3045]]], grad_fn=<ViewBackward0>)

In [9]:
q_head = q.view(batch_size, num_tokens, num_heads, head_dim)
k_head = k.view(batch_size, num_tokens, num_heads, head_dim)
v_head = v.view(batch_size, num_tokens, num_heads, head_dim)
print(q_head.shape)
q_head

torch.Size([2, 3, 2, 4])


tensor([[[[ 0.0868, -0.3236,  0.1107,  0.1021],
          [ 0.3104, -0.5453,  0.3435,  0.1907]],

         [[ 0.3221, -0.5190,  0.1238,  0.2708],
          [ 0.1253, -0.1416, -0.1077,  0.0083]],

         [[-0.0153, -0.5436,  0.5184,  0.4468],
          [ 0.1432, -0.4153,  0.1642,  0.2217]]],


        [[[ 0.3290, -0.5475,  0.2260,  0.2035],
          [-0.0223, -0.2448,  0.0392,  0.2375]],

         [[ 0.0569, -0.4688,  0.3325,  0.3095],
          [ 0.0971, -0.3771,  0.1712,  0.1964]],

         [[ 0.0776, -0.4909,  0.0411,  0.2400],
          [ 0.3649, -0.5768,  0.4504,  0.3045]]]], grad_fn=<ViewBackward0>)

In [10]:
q_head = q_head.transpose(1, 2)
k_head = k_head.transpose(1, 2)
v_head = v_head.transpose(1, 2)
print(q_head.shape)
print("(batch_size, num_heads, num_tokens, head_dim)")
q_head

torch.Size([2, 2, 3, 4])
(batch_size, num_heads, num_tokens, head_dim)


tensor([[[[ 0.0868, -0.3236,  0.1107,  0.1021],
          [ 0.3221, -0.5190,  0.1238,  0.2708],
          [-0.0153, -0.5436,  0.5184,  0.4468]],

         [[ 0.3104, -0.5453,  0.3435,  0.1907],
          [ 0.1253, -0.1416, -0.1077,  0.0083],
          [ 0.1432, -0.4153,  0.1642,  0.2217]]],


        [[[ 0.3290, -0.5475,  0.2260,  0.2035],
          [ 0.0569, -0.4688,  0.3325,  0.3095],
          [ 0.0776, -0.4909,  0.0411,  0.2400]],

         [[-0.0223, -0.2448,  0.0392,  0.2375],
          [ 0.0971, -0.3771,  0.1712,  0.1964],
          [ 0.3649, -0.5768,  0.4504,  0.3045]]]],
       grad_fn=<TransposeBackward0>)

In [11]:
k_t = k_head.transpose(-2, -1)
print(k_t.shape)
k_t

torch.Size([2, 2, 4, 3])


tensor([[[[-4.6698e-01, -4.4905e-01, -3.1901e-01],
          [ 4.8353e-01,  2.0450e-01,  5.9131e-01],
          [ 4.1471e-01,  7.2357e-02,  2.6924e-01],
          [-2.2168e-01, -1.7988e-01, -6.1126e-03]],

         [[ 3.3913e-02,  2.9812e-01, -2.2248e-01],
          [-5.8053e-01, -1.9571e-01, -4.0214e-01],
          [ 2.6886e-01,  3.7796e-01,  4.0605e-01],
          [ 7.8179e-02,  2.5562e-01, -2.9737e-01]]],


        [[[-5.7556e-01, -6.2368e-01, -3.2519e-01],
          [ 1.4396e-01,  3.5102e-01,  5.7178e-01],
          [ 3.1898e-01,  4.2365e-01,  3.9865e-01],
          [ 1.3815e-04,  3.8009e-03, -2.1766e-01]],

         [[ 2.0081e-01,  1.3511e-01, -8.9486e-02],
          [-1.4225e-01, -2.1513e-01, -7.7301e-01],
          [ 4.1049e-01,  4.9382e-01,  3.1264e-01],
          [ 1.6246e-01,  7.8429e-02, -9.1709e-02]]]],
       grad_fn=<TransposeBackward0>)

In [14]:
sim1 = (q_head @ k_t)/math.sqrt(head_dim)
print(sim1.shape)
print("(batch, num_heads, num_queries, num_keys)")
sim1
# Attention Scores Matrix

torch.Size([2, 2, 3, 3])
(batch, num_heads, num_queries, num_keys)


tensor([[[[-0.0869, -0.0578, -0.0949],
          [-0.2050, -0.1453, -0.1890],
          [-0.0699, -0.0736, -0.0898]],

         [[ 0.2172,  0.1889,  0.1165],
          [ 0.0291,  0.0132, -0.0086],
          [ 0.1537,  0.1213,  0.0680]]],


        [[[-0.0980, -0.1504, -0.1871],
          [ 0.0029, -0.0290, -0.1107],
          [-0.0511, -0.1012, -0.1709]],

         [[ 0.0425,  0.0438,  0.0908],
          [ 0.0877,  0.0971,  0.1592],
          [ 0.1949,  0.2099,  0.2631]]]], grad_fn=<DivBackward0>)

In [13]:
# Creates a matrix of shape (3, 3) where lower triangle is 1, rest is 0
mask = torch.tril(torch.ones(num_tokens, num_tokens))
mask

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [15]:
sim1 = sim1.masked_fill(mask == 0, float('-inf'))
sim1

tensor([[[[-0.0869,    -inf,    -inf],
          [-0.2050, -0.1453,    -inf],
          [-0.0699, -0.0736, -0.0898]],

         [[ 0.2172,    -inf,    -inf],
          [ 0.0291,  0.0132,    -inf],
          [ 0.1537,  0.1213,  0.0680]]],


        [[[-0.0980,    -inf,    -inf],
          [ 0.0029, -0.0290,    -inf],
          [-0.0511, -0.1012, -0.1709]],

         [[ 0.0425,    -inf,    -inf],
          [ 0.0877,  0.0971,    -inf],
          [ 0.1949,  0.2099,  0.2631]]]], grad_fn=<MaskedFillBackward0>)

In [16]:
sim2 = F.softmax(sim1, dim=-1)
print(sim2.shape)
print("(num_batches, num_heads, num_queries, num_keys)")
sim2
# Attention Weights matrix
# Basically building a neural network for value vectors

torch.Size([2, 2, 3, 3])
(num_batches, num_heads, num_queries, num_keys)


tensor([[[[1.0000, 0.0000, 0.0000],
          [0.4851, 0.5149, 0.0000],
          [0.3360, 0.3347, 0.3293]],

         [[1.0000, 0.0000, 0.0000],
          [0.5040, 0.4960, 0.0000],
          [0.3465, 0.3355, 0.3180]]],


        [[[1.0000, 0.0000, 0.0000],
          [0.5080, 0.4920, 0.0000],
          [0.3523, 0.3351, 0.3126]],

         [[1.0000, 0.0000, 0.0000],
          [0.4976, 0.5024, 0.0000],
          [0.3241, 0.3290, 0.3470]]]], grad_fn=<SoftmaxBackward0>)

In [17]:
print(v_head.shape)
v_head # Value vectors

torch.Size([2, 2, 3, 4])


tensor([[[[-0.3321, -0.0577,  0.2162,  0.7315],
          [-0.4296, -0.0016,  0.1364,  0.7201],
          [-0.1274, -0.2946,  0.2941,  0.7099]],

         [[ 1.0030,  0.1316, -0.1779, -0.0271],
          [ 0.8849,  0.1411, -0.1703, -0.4948],
          [ 0.9492,  0.2911, -0.1859, -0.2657]]],


        [[[-0.3810,  0.0834,  0.2192,  0.6244],
          [-0.2666, -0.0256,  0.1252,  0.5439],
          [-0.4468, -0.0789,  0.4419,  1.0323]],

         [[ 0.8221, -0.0801,  0.0626, -0.2685],
          [ 0.9129, -0.0030,  0.0230, -0.2487],
          [ 1.1715,  0.2679, -0.2960, -0.0922]]]],
       grad_fn=<TransposeBackward0>)

In [18]:
sim3 = sim2 @ v_head
print(sim3.shape)
sim3 
# Applying Attention Weights to value vectors
# Essentially, passing value vectors through a neural network

torch.Size([2, 2, 3, 4])


tensor([[[[-0.3321, -0.0577,  0.2162,  0.7315],
          [-0.3823, -0.0288,  0.1751,  0.7256],
          [-0.2973, -0.1169,  0.2152,  0.7206]],

         [[ 1.0030,  0.1316, -0.1779, -0.0271],
          [ 0.9444,  0.1363, -0.1741, -0.2591],
          [ 0.9463,  0.1855, -0.1779, -0.2599]]],


        [[[-0.3810,  0.0834,  0.2192,  0.6244],
          [-0.3247,  0.0297,  0.1729,  0.5848],
          [-0.3632, -0.0039,  0.2573,  0.7249]],

         [[ 0.8221, -0.0801,  0.0626, -0.2685],
          [ 0.8677, -0.0414,  0.0427, -0.2586],
          [ 0.9732,  0.0660, -0.0748, -0.2008]]]],
       grad_fn=<UnsafeViewBackward0>)

In [19]:
sim3 = sim3.transpose(1, 2).contiguous()
sim3 = sim3.view(2, 3, 8)
print(sim3.shape)
sim3

torch.Size([2, 3, 8])


tensor([[[-0.3321, -0.0577,  0.2162,  0.7315,  1.0030,  0.1316, -0.1779,
          -0.0271],
         [-0.3823, -0.0288,  0.1751,  0.7256,  0.9444,  0.1363, -0.1741,
          -0.2591],
         [-0.2973, -0.1169,  0.2152,  0.7206,  0.9463,  0.1855, -0.1779,
          -0.2599]],

        [[-0.3810,  0.0834,  0.2192,  0.6244,  0.8221, -0.0801,  0.0626,
          -0.2685],
         [-0.3247,  0.0297,  0.1729,  0.5848,  0.8677, -0.0414,  0.0427,
          -0.2586],
         [-0.3632, -0.0039,  0.2573,  0.7249,  0.9732,  0.0660, -0.0748,
          -0.2008]]], grad_fn=<ViewBackward0>)

In [20]:
output = w_o(sim3)
# Projecting from key_dimension back to d_model (up projection)
print(output.shape)
output
# We have output tensor of the same dimension as our input. Ready to be passed on to the next layer.

torch.Size([2, 3, 8])


tensor([[[-0.0364, -0.4423, -0.2816,  0.5287, -0.1040, -0.4499, -0.2753,
           0.3158],
         [-0.0856, -0.5068, -0.3056,  0.5899, -0.1156, -0.4258, -0.2548,
           0.3607],
         [-0.0951, -0.4716, -0.3136,  0.6179, -0.0923, -0.4250, -0.2445,
           0.3627]],

        [[-0.0873, -0.4385, -0.1356,  0.5474, -0.1959, -0.3271, -0.3039,
           0.3807],
         [-0.1356, -0.4132, -0.1654,  0.5695, -0.1627, -0.3386, -0.2938,
           0.3604],
         [-0.0689, -0.4801, -0.2508,  0.5737, -0.1429, -0.4012, -0.3033,
           0.3858]]], grad_fn=<ViewBackward0>)

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        self.w_o = nn.Linear(self.d_model, self.d_model)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:

        # Checking if input has correct embedding dimension
        if input.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Creating a matrix where lower triangle is 1, rest is 0
        mask = torch.tril(torch.ones(num_tokens, num_tokens))
        # Dimension: (num_queries, num_keys)

        # Applying the mask to our attention scores
        sim1 = sim1.masked_fill(mask == 0, float('-inf'))

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [25]:
dummy_input = torch.rand(2, 3, 8)
dummy_input

tensor([[[0.3069, 0.1549, 0.0495, 0.7172, 0.6932, 0.9086, 0.1293, 0.5317],
         [0.9207, 0.0958, 0.8958, 0.3667, 0.4540, 0.4245, 0.2060, 0.7108],
         [0.0721, 0.7647, 0.0829, 0.9812, 0.1218, 0.5794, 0.6129, 0.1135]],

        [[0.5831, 0.6498, 0.7320, 0.3865, 0.7640, 0.9504, 0.5241, 0.5493],
         [0.2395, 0.7765, 0.8523, 0.8511, 0.5602, 0.2553, 0.9944, 0.7411],
         [0.5304, 0.5764, 0.4463, 0.6189, 0.4484, 0.3833, 0.4378, 0.7112]]])

In [26]:
mha_layer = MultiHeadAttention(d_model=8, num_heads=2)

In [28]:
output = mha_layer(dummy_input)
output

tensor([[[ 0.7093,  0.0128,  0.5221,  0.4929,  0.1943, -0.4859,  0.5271,
          -0.4000],
         [ 0.6298,  0.1583,  0.4258,  0.4268,  0.2167, -0.4731,  0.5225,
          -0.2356],
         [ 0.5603,  0.0884,  0.3903,  0.4066,  0.1808, -0.4643,  0.5414,
          -0.2109]],

        [[ 0.7068,  0.2315,  0.4867,  0.4895,  0.2588, -0.5168,  0.5093,
          -0.2594],
         [ 0.6173,  0.2368,  0.3906,  0.4118,  0.2478, -0.4703,  0.5061,
          -0.1760],
         [ 0.5975,  0.2249,  0.3828,  0.3955,  0.2491, -0.4831,  0.5256,
          -0.1862]]], grad_fn=<ViewBackward0>)